In [1]:
#r "C:\Users\miao\Documents\Git\BoSSS-FSI-Jupyter\internal\src\private-bek\LSS\bin\Debug\net6.0\BoSSSpad.dll"
#r "C:\Users\miao\Documents\Git\BoSSS-FSI-Jupyter\internal\src\private-bek\LSS\bin\Debug\net6.0\LSS.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XdgTimestepping;
Init();

In [2]:
using BoSSS.Foundation.Grid.Classic;
using ilPSP.Utils;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;
using BoSSS.Foundation.Grid;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.Control;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using LSS;
//using ZLS;
using LSS.ControlFiles;
using LSS.Equations;

In [3]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client @C:\Users\miao\Documents\Database
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC2, @\\dc1\userspace\miao\cluster"
2,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC2, @\\dc1\userspace\miao\cluster"


In [4]:
static var myBatch = ExecutionQueues[0];
//static var myDb = myBatch.CreateOrOpenCompatibleDatabase("C:/Users/miao/AppData/Local/BoSSS-LocalJobs/Substrate");
//static var myDb = myBatch.CreateOrOpenCompatibleDatabase("C:/Users/miao/Substrate");
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("Substrate");

In [5]:
BoSSSshell.WorkflowMgm.Init("Substrate");

Project name is set to 'Substrate'.
Default Execution queue is chosen for the database.
Opening existing database 'C:\Users\miao\Documents\Database\Substrate'.


In [6]:
GetDefaultQueue()

DeploymentBaseDirectory,C:\Users\miao\Documents\Database
DeployRuntime,False
RuntimeLocation,<null>
Name,<null>
DotnetRuntime,dotnet
BatchInstructionDir,<null>
AllowedDatabasesPaths,"[ C:\Users\miao\Documents\Database, C:\ ]"


In [7]:
BoSSSshell.WorkflowMgm.AllJobs

In [8]:
wmg.DefaultDatabase

{ Session Count = 51; Grid Count = 99; Path = C:\Users\miao\Documents\Database\Substrate }

In [9]:
databases

#0: { Session Count = 51; Grid Count = 99; Path = C:\Users\miao\Documents\Database\Substrate }


In [10]:
//static var myDb = OpenOrCreateDatabase(@"C:\Databases\BoSSS_DB");
//BoSSSshell.WorkflowMgm.Init("HeatedSquareCavity");

## Create grid

In [11]:
public static class GridFactory {
    public static double[] GetXNodes(int kelem) { 
        var xNodes = GenericBlas.Linspace(-2, 2, kelem + 1);
        return xNodes;
    }
 
    static double[] GetYNodes(int kelem) {
        double[] yNodes =  GenericBlas.Linspace(-2 + 0.3 * 2.0 / kelem, 0.3 * 2.0 / kelem, kelem / 2 + 1);
        return yNodes;
    }
 
    public static Grid2D GenerateGrid(int kelem) { 
        var xNodes = GetXNodes(kelem);
        var yNodes = GetYNodes(kelem);
        var grd    = Grid2D.Cartesian2DGrid(xNodes, yNodes);
        grd.EdgeTagNames.Add(1, "Wall");
        //grd.EdgeTagNames.Add(2, "Wall_tempfixed_bottom");
        grd.DefineEdgeTags( delegate (double[] X) {
            double x = X[0];
            double y = X[1];

            //top cold wall
            if (Math.Abs(y - 0.3 * 2.0 / kelem ) < 1e-8)
                return 1;

            //bottom hot Wall
            if (Math.Abs(y + 2 - 0.3 * 2.0 / kelem ) < 1e-8)
                return 1;

            if (Math.Abs(x + 2 ) < 1e-8)
                return 1;

            if (Math.Abs(x - 2 ) < 1e-8)
                return 1;
            

            else throw new ArgumentOutOfRangeException();
        });
        
        bool force = true; 
        myDb.SaveGrid(ref grd, force);
        
        return grd;
     }
 
 }

In [12]:
public static class BoundaryValueFactory { 

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double VelX(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double VelY(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_VelX() {
        return new Formula("BoundaryValues.VelX", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_VelY(){
        return new Formula("BoundaryValues.VelY", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [13]:
public static LSSControl Aland(int p = 2, int kelem = 16) {
    LSSControl C = new LSSControl(p, BoundaryCollection.Line(new Vector(-2, -0.0001), new Vector(2, -0.000001), 4 * (kelem * 9 + 1)));
    C.HasContactLine = true;
    C.FreePressureMeanValue = false;
    //C.ImmediatePlotPeriod = 1;
    //C.SuperSampling = 3;
    C.AgglomerationThreshold = 0.1;
    C.NoOfMultigridLevels = 1;

    int D = 2;


    MultidimensionalArray BoundaryForce(Vector X) {
        MultidimensionalArray nb = MultidimensionalArray.Create(D, D + 1);
        if(Math.Abs(X[0]) <= 0.4) {
            nb[0, D] = 0.115 + 0.03;
            nb[1, D] = 0.115 + 0.03;
        }
        return nb;
    }
    C.NeumannBoundary.Set(new BoundaryConditionFromFunction<MultidimensionalArray>(BoundaryForce));
    Vector cf = new Vector(D);
    cf[1] = -0.046;
    C.ContactForce.Set(new BoundaryConditionFromFunction<Vector>(X => cf));

    double phi(double x, double y) {
        return -x * x + 0.16;
    }

    C.ContactLevelSet = phi;

    // basic database options
    // ======================
    #region db

    C.savetodb = true;
    //C.DbPath = @"C:\Users\miao\Substrate";
    C.ProjectName = "Substrate";
    C.SessionName = "Substrate local";
    C.ProjectDescription = "Substrate running on pc";
    C.ContinueOnIoError = false;
    //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
    //C.PostprocessingModules.Add(new MovingContactLineLogging());

    #endregion


    // DG degrees
    // ==========
    #region degrees
    //C.SetDGdegree(p);

    #endregion


    // Physical Parameters
    // ===================
    #region physics
    double scale = 4.4175e-4;
    C.Material = new NavierCauchy() {
        Density = 1000 * scale * scale * scale,
        Lame2 = 1000 * scale,
        Viscosity = 10 * scale,
    };

    #endregion


    // grid generation
    // ===============
    #region grid

    C.SetGrid(GridFactory.GenerateGrid(kelem));

    C.AddBoundaryValue("Wall", "VelocityX", BoundaryValueFactory.Get_VelX());
    C.AddBoundaryValue("Wall", "VelocityY", BoundaryValueFactory.Get_VelY());
    //C.AddBoundaryValue("wall");

    #endregion


    // Initial Values
    // ==============
    #region init

    //C.AddInitialValue("Phi", new Formula("X => X[0] -0.73"));

    #endregion

    // misc. solver options
    // ====================
    #region solver

    //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
    //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
    //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;

    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    //C.Solver_ConvergenceCriterion = 1e-8;
    C.FreePressureMeanValue = false;
    #endregion


    // Timestepping
    // ============
    #region time

    //C.CheckJumpConditions = true;
    C.NoOfTimesteps = 10;
    double dt = 1e-2;
    C.dtMax = dt;
    C.dtMin = dt;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;

    C.AdaptiveMeshRefinement = false;

    #endregion

    return C;
}

In [14]:
var ctest = Aland(2,16);

We are here2
We are here2
We are here2
Grid Edge Tags changed.


In [15]:
ctest.Geometry.points.Take(3)

index,value
0,"[ -2, -0.0001 ]"
1,"[ -1.9933218192285551, -9.983471502590674E-05 ]"
2,"[ -1.9861830742659756, -9.96580310880829E-05 ]"


In [16]:
ctest.Geometry.knots.Take(4)

[ -1, -0.9965457685664939, -0.9930915371329879, -0.9896373056994818 ]

In [17]:
string ctestString = ctest.Serialize();

In [18]:
var ctest_recover = AppControl.Deserialize(ctestString) as LSSControl;

In [19]:
ctest_recover.Geometry.knots.Take(4)

[ -1, -0.9965457685664939, -0.9930915371329879, -0.9896373056994818 ]

In [20]:
ctest_recover.Geometry.points.Take(4)

index,value
0,[ ]
1,[ ]
2,[ ]
3,[ ]


In [21]:
using Newtonsoft.Json;

In [22]:
using System.Reflection;

In [4]:
class __KnownTypesBinder : Newtonsoft.Json.Serialization.DefaultSerializationBinder {

    internal __KnownTypesBinder(Type entry) {

        HashSet<Assembly> assiList = new HashSet<Assembly>();
        GetAllAssemblies(entry.Assembly, assiList);

        foreach(var a in assiList) {
            var tt = new Dictionary<string, Type>();
            knownTypes.Add(a.GetName().Name, tt);
            foreach(var t in a.GetExportedTypes()) {
                tt.Add(t.FullName, t);
            }
        }
    }


    /// <summary>
    /// Recursive collection of all dependencies of some assembly.
    /// </summary>
    /// <param name="a"></param>
    /// <param name="assiList">
    /// Output, list where all dependent assemblies are collected.
    /// </param>
    private static void GetAllAssemblies(Assembly a, HashSet<Assembly> assiList) {
        if(assiList.Contains(a))
            return;
        assiList.Add(a);

        string fileName = Path.GetFileName(a.Location);
        var allMatch = assiList.Where(_a => Path.GetFileName(_a.Location).Equals(fileName)).ToArray();
        if(allMatch.Length > 1) {
            throw new ApplicationException("internal error in assembly collection.");
        }


        foreach(AssemblyName b in a.GetReferencedAssemblies()) {
            Assembly na;
            try {
                na = Assembly.Load(b);
            } catch(FileNotFoundException) {
                //string[] AssiFiles = ArrayTools.Cat(Directory.GetFiles(SearchPath, b.Name + ".dll"), Directory.GetFiles(SearchPath, b.Name + ".exe"));
                //if(AssiFiles.Length != 1) {
                //    //throw new FileNotFoundException("Unable to locate assembly '" + b.Name + "'.");
                //    Console.WriteLine("Skipping: " + b.Name);
                //    continue;
                //}
                //na = Assembly.LoadFile(AssiFiles[0]);
                continue;
            } catch (FileLoadException) {
                //string[] AssiFiles = ArrayTools.Cat(Directory.GetFiles(SearchPath, b.Name + ".dll"), Directory.GetFiles(SearchPath, b.Name + ".exe"));
                //if(AssiFiles.Length != 1) {
                //    //throw new FileNotFoundException("Unable to locate assembly '" + b.Name + "'.");
                //    Console.WriteLine("Skipping: " + b.Name);
                //    continue;
                //}
                //na = Assembly.LoadFile(AssiFiles[0]);
                continue;
            }

            GetAllAssemblies(na, assiList);
        }
    }


    Dictionary<string, Dictionary<string, Type>> knownTypes = new Dictionary<string, Dictionary<string, Type>>();

    /*
    public IList<Type> KnownTypes { get; set; }

    public Type BindToType(string assemblyName, string typeName) {
        return KnownTypes.SingleOrDefault(t => t.Name == typeName);
    }

    public void BindToName(Type serializedType, out string assemblyName, out string typeName) {
        assemblyName = null;
        typeName = serializedType.Name;
    }
    */
    public override Type BindToType(string assemblyName, string typeName) {
        //Console.WriteLine("Type lookup: " + assemblyName + "+" + typeName);
        var dd = knownTypes[assemblyName];
        var tt = dd[typeName];


        return tt;
    }
}


In [5]:
static class Test { 
    static public string Serialize(object o) {
            JsonSerializer formatter = new JsonSerializer() {
                NullValueHandling = NullValueHandling.Ignore,
                TypeNameHandling = TypeNameHandling.Auto,
                ConstructorHandling = ConstructorHandling.AllowNonPublicDefaultConstructor,
                ReferenceLoopHandling = ReferenceLoopHandling.Error,
                Formatting = Formatting.Indented
//                ObjectCreationHandling = ObjectCreationHandling.
            };
                        
            using(var tw = new StringWriter()) {
                tw.WriteLine(o.GetType().AssemblyQualifiedName);
                using(JsonWriter writer = new JsonTextWriter(tw)) {  // Alternative: binary writer: BsonWriter
                    formatter.Serialize(writer, o);
                }

                string Ret = tw.ToString();
                return Ret;
            }
            
        }
    public static object Deserialize(string Str, Newtonsoft.Json.Serialization.ISerializationBinder binder) {
        JsonSerializer formatter = new JsonSerializer() {
            NullValueHandling = NullValueHandling.Ignore,
            TypeNameHandling = TypeNameHandling.Auto,
            ConstructorHandling = ConstructorHandling.AllowNonPublicDefaultConstructor,
            ReferenceLoopHandling = ReferenceLoopHandling.Error
        };



        using(var tr = new StringReader(Str)) {
            string typeName = tr.ReadLine();
            Type ControlObjectType = Type.GetType(typeName);

            if(binder != null)
                formatter.SerializationBinder = binder;
            else
                formatter.SerializationBinder = new __KnownTypesBinder(ControlObjectType);


            using(JsonReader reader = new JsonTextReader(tr)) {
                var obj = formatter.Deserialize(reader, ControlObjectType);

                return obj;
            }
        
        }
            

    }
}


In [21]:
//string ss = Test.Serialize(new Vector[] { new Vector(1,2,3), new Vector(4,5,6),  });
//string ss = Test.Serialize( new Vector(1,2,3) );
string ss = Test.Serialize( new double[][] { new double[] {1,2,3}, new double[] {1,2,6} });
ss

System.Double[][], System.Private.CoreLib, Version=7.0.0.0, Culture=neutral, PublicKeyToken=7cec85d7bea7798e
[
  [
    1.0,
    2.0,
    3.0
  ],
  [
    1.0,
    2.0,
    6.0
  ]
]

In [22]:
Test.Deserialize(ss, null)

index,value
0,"[ 1, 2, 3 ]"
1,"[ 1, 2, 6 ]"


In [17]:
Test.Deserialize(ss, new __KnownTypesBinder(typeof(ilPSP.Vector)))

index,value
0,"[ 1, 2, 3 ]"
1,"[ 1, 2, 6 ]"


In [19]:
//var ctrl = LSS.ControlFiles.Beam2.Unsteady();

In [20]:
//ctrl.VerifyEx()

## Send and run jobs

In [16]:
    var C_CTRLFILE = Aland(2,16);//Create control file.

We are here2
We are here2
We are here2
Grid Edge Tags changed.


In [17]:
    var JobLocal = C_CTRLFILE.CreateJob();

In [18]:
JobLocal.Status

PreActivation

In [19]:
    JobLocal.NumberOfMPIProcs = 1;
    JobLocal.Activate();
    JobLocal.ShowOutput();
    //BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(12*60*60);

Deploying job Substrate local ... 


Error: System.ArgumentException: Unable to serialize/deserialize 'Geometry.points[0][0]' correctly. (Missing a DataMemberAtribute in control class?)
   at BoSSS.Application.BoSSSpad.AppControlExtensions.VerifyEx(AppControl ctrl) in C:\Users\miao\Documents\Git\BoSSS-FSI-Jupyter\public\src\L4-application\BoSSSpad\AppControlExtensions.cs:line 322
   at BoSSS.Application.BoSSSpad.Job.FiddleControlFile(BatchProcessorClient bpc) in C:\Users\miao\Documents\Git\BoSSS-FSI-Jupyter\public\src\L4-application\BoSSSpad\Job.cs:line 464
   at BoSSS.Application.BoSSSpad.Job.Reactivate() in C:\Users\miao\Documents\Git\BoSSS-FSI-Jupyter\public\src\L4-application\BoSSSpad\Job.cs:line 1484
   at BoSSS.Application.BoSSSpad.Job.Activate(BatchProcessorClient bpc) in C:\Users\miao\Documents\Git\BoSSS-FSI-Jupyter\public\src\L4-application\BoSSSpad\Job.cs:line 1446
   at BoSSS.Application.BoSSSpad.Job.Activate() in C:\Users\miao\Documents\Git\BoSSS-FSI-Jupyter\public\src\L4-application\BoSSSpad\Job.cs:line 

In [ ]:
//JobLocal.Stdout

In [11]:
wmg.Sessions

#0: Substrate	Substrate local*	8/6/2023 10:05:54 PM	ca86fa35...
#1: Substrate	Substrate local*	8/6/2023 10:03:05 PM	2b36fc35...
#2: Substrate	Substrate local	8/6/2023 10:02:14 PM	67ec30fe...
#3: Substrate	Substrate local*	8/6/2023 9:48:25 PM	de8de0fb...


In [25]:
//var outPath = wmg.Sessions[0].Export().WithSupersampling(2).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Substrate__Substrate local__a564692a-bf19-4d37-9ffc-f5314516c046


## Post processing

In [30]:
var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [37]:
var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

Error: (1,3): error CS1501: No overload for method 'Evaluate' takes 0 arguments